# AgroSele — Cross-Encoder

Quinta variante, e a primeira que não é bi-encoder: até agora, pergunta e
resposta sempre viravam embeddings SEPARADOS, calculados de forma
independente. Aqui, o BERT recebe os dois textos **juntos**, na mesma
sequência: `[CLS] pergunta [SEP] candidata [SEP]`. Isso permite **atenção
cruzada** — cada palavra da pergunta pode "olhar" diretamente pra cada
palavra da candidata em todas as camadas do Transformer, antes de qualquer
decisão de classificação. É o jeito mais "caro" de comparar dois textos,
mas costuma dar mais qualidade.

**Sobre esse notebook**: extrair o vetor `pooler_output` de cada par
(pergunta, candidata) levou **~2h15min em CPU** (38.263 pares, sem
possibilidade de cache entre pares diferentes — cada par é uma sequência
única). Não faz sentido reprocessar isso toda vez que o notebook roda, então
aqui eu **carrego o cache já extraído** e só treino a cabeça de
classificação por cima (isso sim é rápido, features já prontas).

In [1]:
import itertools
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)

C:\Users\frede\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Por que não dá pra cachear embeddings separados aqui

No bi-encoder, `emb(pergunta)` não depende de qual candidata está sendo
comparada — dá pra calcular uma vez só e reaproveitar em 50 comparações
diferentes. Aqui não: o vetor que sai do BERT já é a representação **do par
inteiro**, então preciso de um vetor por PAR, não um por texto. Para
2.307 perguntas de treino (8 negativos cada) + dev completo (50×50) + teste
completo (300×50), isso dá 38.263 pares únicos.

O código de extração (`extrair_features_pares.py`, roda fora deste
notebook) é basicamente:

```python
entrada = tokenizador(lista_perguntas, lista_respostas, return_tensors="pt",
                       truncation=True, max_length=256, padding=True)
saida = bert(**entrada)
vetor_do_par = saida.pooler_output  # (lote, 768) -- ja "e" a representacao do par
```

Repare que o tokenizer recebe DOIS textos posicionalmente — isso monta a
sequência `[CLS] pergunta [SEP] candidata [SEP]` automaticamente, com os
`token_type_ids` certos pra marcar qual pedaço é pergunta e qual é resposta.

In [2]:
caminho_cache = "cache/pair_features_crossencoder.pt"
print(f"Carregando cache de features do cross-encoder: {caminho_cache}")
cache = torch.load(caminho_cache, weights_only=False)
print(f"{len(cache)} pares no cache | dim={next(iter(cache.values())).shape[0]}")

print("Carregando splits oficiais do MilkQA...")
ds = load_dataset("eduagarcia/MilkQA")
conjunto_treino, conjunto_dev, conjunto_teste = ds["train"], ds["dev"], ds["test"]
print(f"treino={len(conjunto_treino)} | dev={len(conjunto_dev)} | teste={len(conjunto_teste)}")

Carregando cache de features do cross-encoder: cache/pair_features_crossencoder.pt


38263 pares no cache | dim=768
Carregando splits oficiais do MilkQA...


treino=2307 | dev=50 | teste=300


## 1.1 Verificação: o cache é fiel ao BERT de verdade?

Mesma pergunta legítima de sempre: como saber que esse cache não foi
inventado? Abaixo, pego 5 pares (pergunta, candidata) ao acaso, rodo o
BERTimbau **ao vivo** só para eles (a sequência conjunta de novo), e
comparo com o vetor salvo no cache. O modelo está congelado e a extração é
determinística, então o recálculo tem que bater.

In [3]:
import csv as _csv
import random as _random

print("Verificando fidelidade do cache: recalculando 5 pares ao vivo...")
from transformers import AutoModel, AutoTokenizer

_texto_resposta_verif, _texto_pergunta_verif = {}, {}
with open("../selecao-resposta-milkqa-finetune/datasets/corpus.csv", encoding="utf-8") as f:
    for _linha in _csv.DictReader(f):
        _texto_resposta_verif[_linha["id"]] = _linha["text"]
with open("../selecao-resposta-milkqa-finetune/datasets/queries.csv", encoding="utf-8") as f:
    for _linha in _csv.DictReader(f):
        _texto_pergunta_verif[_linha["id"]] = _linha["text"]

_chaves_amostra = _random.Random(123).sample(list(cache.keys()), 5)

_tok_verif = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
_modelo_verif = AutoModel.from_pretrained("neuralmind/bert-base-portuguese-cased")
_modelo_verif.eval()

_diffs = []
with torch.no_grad():
    for _chave in _chaves_amostra:
        _id_pergunta, _id_resposta = _chave.split("||")
        _entrada = _tok_verif([_texto_pergunta_verif[_id_pergunta]], [_texto_resposta_verif[_id_resposta]],
                               return_tensors="pt", truncation=True, max_length=256, padding=True)
        _saida = _modelo_verif(**_entrada)
        _vetor_ao_vivo = _saida.pooler_output[0]
        _vetor_cache = cache[_chave]
        _diff = (_vetor_ao_vivo - _vetor_cache).abs().max().item()
        _diffs.append(_diff)
        print(f"  par {_chave}: diferenca maxima entre cache e recalculo ao vivo = {_diff:.2e}")

assert max(_diffs) < 1e-4, "cache NAO bate com o recalculo ao vivo do BERT!"
print("Cache confirmado: os vetores salvos batem com o recalculo ao vivo do BERTimbau.")
del _modelo_verif, _tok_verif

Verificando fidelidade do cache: recalculando 5 pares ao vivo...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8847.24it/s]


[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  par 1112||3847: diferenca maxima entre cache e recalculo ao vivo = 3.58e-07


  par 15801||17525: diferenca maxima entre cache e recalculo ao vivo = 2.98e-07


  par 14181||7915: diferenca maxima entre cache e recalculo ao vivo = 2.98e-07


  par 25461||329: diferenca maxima entre cache e recalculo ao vivo = 2.98e-07


  par 9988||9301: diferenca maxima entre cache e recalculo ao vivo = 1.94e-07
Cache confirmado: os vetores salvos batem com o recalculo ao vivo do BERTimbau.


## 2. Cabeça de classificação sobre o vetor do par

Repare que aqui o vetor de entrada é só **768 dimensões** (o
`pooler_output` já representa o par inteiro) — bem menor que as 3072/3074
dimensões das variantes bi-encoder, porque não há `q_emb`/`a_emb`
separados pra concatenar, subtrair e multiplicar. Toda a interação entre
pergunta e candidata já aconteceu DENTRO do BERT, via atenção cruzada.

In [4]:
class CabecaCrossEncoder(nn.Module):
    """Cabeca de classificacao sobre o vetor [CLS]/pooler do par ja
    codificado em conjunto pelo BERT (768d)."""

    def __init__(self, dim_entrada, ocultas, dropout):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(dim_entrada, ocultas), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(ocultas, 1),
        )

    def forward(self, x):
        return self.rede(x).squeeze(-1)


def montar_pares_treino(conjunto, n_perguntas, n_negativos):
    linhas = list(conjunto)
    rng = random.Random(SEMENTE)
    linhas = rng.sample(linhas, min(n_perguntas, len(linhas)))
    pares = []
    for linha in linhas:
        id_pergunta, id_certa = linha["query-id"], linha["positive-doc-id"]
        candidatas = [c for c in linha["candidates-ids"] if c != id_certa]
        negativos = rng.sample(candidatas, min(n_negativos, len(candidatas)))
        pares.append((id_pergunta, id_certa, 1))
        for id_neg in negativos:
            pares.append((id_pergunta, id_neg, 0))
    return pares


def montar_X_y(pares, cache):
    X = torch.stack([cache[f"{q}||{c}"] for q, c, _ in pares])
    y = torch.tensor([r for _, _, r in pares], dtype=torch.float32)
    return X, y


def avaliar_ranking(modelo, conjunto, cache):
    modelo.eval()
    lista_acuracia1, lista_mrr = [], []
    with torch.no_grad():
        for linha in conjunto:
            id_pergunta, id_certa, candidatas = linha["query-id"], linha["positive-doc-id"], linha["candidates-ids"]
            features = torch.stack([cache[f"{id_pergunta}||{c}"] for c in candidatas])
            pontuacoes = torch.sigmoid(modelo(features)).numpy()

            ordem = np.argsort(-pontuacoes)
            ids_ranqueados = [candidatas[i] for i in ordem]
            posicao = ids_ranqueados.index(id_certa) + 1
            lista_acuracia1.append(1.0 if posicao == 1 else 0.0)
            lista_mrr.append(1.0 / posicao)
    return float(np.mean(lista_acuracia1)), float(np.mean(lista_mrr))


def treinar_um_modelo(X_treino, y_treino, conjunto_dev, cache, ocultas, dropout, lr):
    torch.manual_seed(SEMENTE)
    modelo = CabecaCrossEncoder(X_treino.shape[1], ocultas, dropout)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    funcao_perda = nn.BCEWithLogitsLoss()

    melhor_mrr, melhor_estado, sem_melhora = -1.0, None, 0
    n = X_treino.shape[0]
    tamanho_lote = 64
    gerador = torch.Generator().manual_seed(SEMENTE)

    for epoca in range(30):
        modelo.train()
        permutacao = torch.randperm(n, generator=gerador)
        for i in range(0, n, tamanho_lote):
            indices = permutacao[i:i + tamanho_lote]
            otimizador.zero_grad()
            logits = modelo(X_treino[indices])
            perda = funcao_perda(logits, y_treino[indices])
            perda.backward()
            otimizador.step()

        _, mrr_dev = avaliar_ranking(modelo, conjunto_dev, cache)
        if mrr_dev > melhor_mrr:
            melhor_mrr, melhor_estado, sem_melhora = mrr_dev, {k: v.clone() for k, v in modelo.state_dict().items()}, 0
        else:
            sem_melhora += 1
            if sem_melhora >= 5:
                break

    modelo.load_state_dict(melhor_estado)
    return modelo, melhor_mrr

## 3. Treino (grid search sobre a cabeça só -- rápido, features fixas)

In [5]:
N_PERGUNTAS_TREINO = 2307
N_NEGATIVOS_TREINO = 8

print(f"Montando pares de treino ({N_PERGUNTAS_TREINO} perguntas, {N_NEGATIVOS_TREINO} negativos)...")
pares_treino = montar_pares_treino(conjunto_treino, N_PERGUNTAS_TREINO, N_NEGATIVOS_TREINO)
X_treino, y_treino = montar_X_y(pares_treino, cache)
print(f"{X_treino.shape[0]} pares ({int(y_treino.sum())} positivos, {int((1 - y_treino).sum())} negativos)")

Montando pares de treino (2307 perguntas, 8 negativos)...


20763 pares (2307 positivos, 18456 negativos)


In [6]:
print("\n===== Grid Search (selecao pelo MRR no dev) =====")
grade = {"ocultas": [64, 128], "dropout": [0.2, 0.4], "lr": [1e-3, 1e-4]}
combinacoes = list(itertools.product(grade["ocultas"], grade["dropout"], grade["lr"]))
resultados = []
melhor_geral = {"mrr": -1.0, "modelo": None, "config": None}
for ocultas, dropout, lr in combinacoes:
    modelo, mrr_dev = treinar_um_modelo(X_treino, y_treino, conjunto_dev, cache, ocultas, dropout, lr)
    acuracia1_dev, _ = avaliar_ranking(modelo, conjunto_dev, cache)
    resultados.append({"ocultas": ocultas, "dropout": dropout, "lr": lr,
                        "mrr_dev": mrr_dev, "acuracia1_dev": acuracia1_dev})
    print(f"  ocultas={ocultas:4d} dropout={dropout:.1f} lr={lr:.0e} "
          f"-> dev MRR={mrr_dev:.4f} Acc@1={acuracia1_dev:.4f}")
    if mrr_dev > melhor_geral["mrr"]:
        melhor_geral = {"mrr": mrr_dev, "modelo": modelo, "config": (ocultas, dropout, lr)}

ocultas, dropout, lr = melhor_geral["config"]
print(f"\nMelhor configuracao: ocultas={ocultas}, dropout={dropout}, lr={lr} (dev MRR={melhor_geral['mrr']:.4f})")


===== Grid Search (selecao pelo MRR no dev) =====


  ocultas=  64 dropout=0.2 lr=1e-03 -> dev MRR=0.8175 Acc@1=0.7600


  ocultas=  64 dropout=0.2 lr=1e-04 -> dev MRR=0.8182 Acc@1=0.7600


  ocultas=  64 dropout=0.4 lr=1e-03 -> dev MRR=0.8175 Acc@1=0.7600


  ocultas=  64 dropout=0.4 lr=1e-04 -> dev MRR=0.8178 Acc@1=0.7600


  ocultas= 128 dropout=0.2 lr=1e-03 -> dev MRR=0.8176 Acc@1=0.7600


  ocultas= 128 dropout=0.2 lr=1e-04 -> dev MRR=0.8179 Acc@1=0.7600


  ocultas= 128 dropout=0.4 lr=1e-03 -> dev MRR=0.8179 Acc@1=0.7600


  ocultas= 128 dropout=0.4 lr=1e-04 -> dev MRR=0.8181 Acc@1=0.7600

Melhor configuracao: ocultas=64, dropout=0.2, lr=0.0001 (dev MRR=0.8182)


In [7]:
print("\n===== Avaliacao final no TESTE (300 perguntas, 50 candidatas cada) =====")
modelo_final = melhor_geral["modelo"]
acuracia1_teste, mrr_teste = avaliar_ranking(modelo_final, conjunto_teste, cache)
print(f"Accuracy@1 (teste, cross-encoder) = {acuracia1_teste:.4f}")
print(f"MRR (teste, cross-encoder)        = {mrr_teste:.4f}")
print("\nComparar com: bi-encoder congelado = 0.570/0.679 | hibrido BERT+BM25 = 0.663/0.753 "
      "| fine-tuning completo = 0.690/0.782")


===== Avaliacao final no TESTE (300 perguntas, 50 candidatas cada) =====


Accuracy@1 (teste, cross-encoder) = 0.6167
MRR (teste, cross-encoder)        = 0.7152

Comparar com: bi-encoder congelado = 0.570/0.679 | hibrido BERT+BM25 = 0.663/0.753 | fine-tuning completo = 0.690/0.782


In [8]:
import os
os.makedirs("checkpoints", exist_ok=True)
torch.save({
    "model_state": modelo_final.state_dict(),
    "ocultas": ocultas, "dropout": dropout, "lr": lr,
    "dim_entrada": X_treino.shape[1],
    "acuracia1_teste": acuracia1_teste, "mrr_teste": mrr_teste,
}, "checkpoints/best_model_crossencoder_notebook.pt")
pd.DataFrame(resultados).sort_values("mrr_dev", ascending=False).to_csv(
    "checkpoints/grid_search_results_notebook.csv", index=False)
print("Checkpoint e grid search salvos em checkpoints/")

Checkpoint e grid search salvos em checkpoints/


## Conclusão

O Cross-Encoder chega em `Accuracy@1 ≈ 0,617` e `MRR ≈ 0,715` — supera o
bi-encoder congelado equivalente (0,570/0,679) **sem nenhum treinamento do
BERT**, só trocando a arquitetura de "dois embeddings separados" pra
"atenção conjunta entre os dois textos". Isso confirma a expectativa da
literatura (cross-encoders > bi-encoders na mesma configuração), ao custo
de não permitir pré-computar/indexar embeddings de candidatas
independentemente da pergunta — cada comparação exige rodar o BERT de novo.
Fica atrás do híbrido (BM25 fundido) e do fine-tuning completo, sugerindo
que, nesta escala de dado, o sinal lexical explícito e o ajuste de pesos
ainda valem mais do que a arquitetura de atenção cruzada isolada.

Ver `notebook_pipeline_multiestagio.ipynb` para como usar esse Cross-Encoder
de forma mais barata em produção, combinando com o BM25.